In [2]:
from tqdm import tqdm
import os
from os import listdir
import time
from random import randint
from os.path import isfile, join

from scipy.stats import skew, kurtosis
import pickle as pkl
 
import gc 
import numpy as np
from scipy import stats
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import KFold

import nibabel as nib
import pydicom as pdm
import nilearn as nl
import nilearn.plotting as nlplt
import h5py

from skimage import feature

import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.animation as anim
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

import seaborn as sns
import imageio
from skimage.transform import resize
from skimage.util import montage

# from IPython.display import Image as show_gif
# from IPython.display import clear_output
# from IPython.display import YouTubeVideo

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn import MSELoss

# !pip install opencv-python==4.6.0.66
# !pip install -U albumentations --no-binary qudida,albumentations
import albumentations as A
# from albumentations.pytorch import ToTensor, ToTensorV2


from albumentations import Compose, HorizontalFlip
# from albumentations.pytorch import ToTensor, ToTensorV2 

import warnings
warnings.simplefilter("ignore")

# Function to Calculate Volume of a Tumor based on mask file

In [4]:
def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET, mask_ET])
    
    return mask 

def get_tumor_slices(mri_mask):
    """
    Extracts slices from a 3D MRI mask where the tumor exists.

    Parameters:
    mri_mask (numpy.ndarray): A 3D NumPy array representing the MRI mask.

    Returns:
    list of numpy.ndarray: A list of 2D slices containing the tumor.
    """
    tumor_slices = []

    # Iterate through each slice
    for i in range(mri_mask.shape[2]):
        _slice = mri_mask[:, :, i]
        
        # Check if the slice contains tumor (non-zero values)
        if np.any(_slice):
            tumor_slices.append(i)

    return tumor_slices

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = 'BraTS-GLI-' + patient_id + '/' + 'BraTS-GLI-' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img 

def calculate_intensity(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img   = read_MRI(dataset, patient_id)
    average_intensity_t1 = np.mean(t1_img)
    average_intensity_tice = np.mean(t1ce_img)
    average_intensity_t2 = np.mean(t2_img)
    average_intensity_flair = np.mean(flair_img)
    
    return {'t1':average_intensity_t1, 
            't1ce':average_intensity_tice, 
            't2':average_intensity_t2, 
            'flair':average_intensity_flair}

def calculate_intensity_mask_only(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img   = read_MRI(dataset, patient_id)
    
    slices = get_tumor_slices(mask_img)
    start_slice = slices[0]
    end_slice = slices[-1]
    
    t1_img = t1_img[:,:, start_slice:end_slice]
    t1ce_img = t1ce_img[:,:, start_slice:end_slice]
    t2_img = t2_img[:,:, start_slice:end_slice]
    flair_img = flair_img[:,:, start_slice:end_slice]
    
    average_intensity_t1 = np.mean(t1_img)
    average_intensity_tice = np.mean(t1ce_img)
    average_intensity_t2 = np.mean(t2_img)
    average_intensity_flair = np.mean(flair_img)
    
    return {'t1':average_intensity_t1, 
            't1ce':average_intensity_tice, 
            't2':average_intensity_t2, 
            'flair':average_intensity_flair}


def calculate_intensity_tumor(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img   = read_MRI(dataset, patient_id)
    
    slices = get_tumor_slices(mask_img)
    start_slice = slices[0]
    end_slice = slices[-1]
    
    t1_img = t1_img[:,:, start_slice:end_slice]
    t1ce_img = t1ce_img[:,:, start_slice:end_slice]
    t2_img = t2_img[:,:, start_slice:end_slice]
    flair_img = flair_img[:,:, start_slice:end_slice]
    
    mask_img = mask_img[:,:, start_slice:end_slice].astype(bool)
    
    average_intensity_t1_tumor = np.mean(t1_img[mask_img])
    average_intensity_tice_tumor = np.mean(t1ce_img[mask_img])
    average_intensity_t2_tumor = np.mean(t2_img[mask_img])
    average_intensity_flair_tumor = np.mean(flair_img[mask_img])
    
    average_intensity_t1_non_tumor = np.mean(t1_img[~mask_img])
    average_intensity_tice_non_tumor = np.mean(t1ce_img[~mask_img])
    average_intensity_t2_non_tumor = np.mean(t2_img[~mask_img])
    average_intensity_flair_non_tumor = np.mean(flair_img[~mask_img])
    
    return {'tumor': {'t1':average_intensity_t1_tumor, 
                    't1ce':average_intensity_tice_tumor, 
                    't2':average_intensity_t2_tumor, 
                    'flair':average_intensity_flair_tumor}, 
            'non-tumor' : {'t1':average_intensity_t1_non_tumor, 
                    't1ce':average_intensity_tice_non_tumor, 
                    't2':average_intensity_t2_non_tumor, 
                    'flair':average_intensity_flair_non_tumor}
           }



def calculate_intensity_tumor(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img = read_MRI(dataset, patient_id)
    
    slices = get_tumor_slices(mask_img)
    start_slice = slices[0]
    end_slice = slices[-1]
    
    t1_img = t1_img[:,:, start_slice:end_slice]
    t1ce_img = t1ce_img[:,:, start_slice:end_slice]
    t2_img = t2_img[:,:, start_slice:end_slice]
    flair_img = flair_img[:,:, start_slice:end_slice]
    
    mask_img = mask_img[:,:, start_slice:end_slice].astype(bool)

    def calculate_stats(image, mask):
        tumor_values = image[mask]
        non_tumor_values = image[~mask]

        tumor_stats = {
            'average_intensity': np.mean(tumor_values),
            'variance': np.var(tumor_values),
            'skewness': skew(tumor_values),
            'kurtosis': kurtosis(tumor_values)
        }

        non_tumor_stats = {
            'average_intensity': np.mean(non_tumor_values),
            'variance': np.var(non_tumor_values),
            'skewness': skew(non_tumor_values),
            'kurtosis': kurtosis(non_tumor_values)
        }

        return tumor_stats, non_tumor_stats

    t1_stats = calculate_stats(t1_img, mask_img)
    t1ce_stats = calculate_stats(t1ce_img, mask_img)
    t2_stats = calculate_stats(t2_img, mask_img)
    flair_stats = calculate_stats(flair_img, mask_img)

    return {
        'tumor': {
            't1': t1_stats[0],
            't1ce': t1ce_stats[0],
            't2': t2_stats[0],
            'flair': flair_stats[0]
        },
        'non-tumor': {
            't1': t1_stats[1],
            't1ce': t1ce_stats[1],
            't2': t2_stats[1],
            'flair': flair_stats[1]
        }
    }


def calculate_intensity_tumor_whole_image(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img = read_MRI(dataset, patient_id)
    
    slices = get_tumor_slices(mask_img)
    start_slice = slices[0]
    end_slice = slices[-1]
    
#     t1_img = t1_img[:,:, start_slice:end_slice]
#     t1ce_img = t1ce_img[:,:, start_slice:end_slice]
#     t2_img = t2_img[:,:, start_slice:end_slice]
#     flair_img = flair_img[:,:, start_slice:end_slice]
    
#     mask_img = mask_img[:,:, start_slice:end_slice].astype(bool)

    def calculate_stats(image, mask):
        tumor_values = image#[mask]
        flattened_image = tumor_values.flatten()
        tumor_stats = {
            'average_intensity': np.mean(tumor_values),
            'variance': np.var(tumor_values),
            'skewness': skew(flattened_image),
            'kurtosis': kurtosis(flattened_image)
        }

        return tumor_stats

    t1_stats = calculate_stats(t1_img, mask_img)
    t1ce_stats = calculate_stats(t1ce_img, mask_img)
    t2_stats = calculate_stats(t2_img, mask_img)
    flair_stats = calculate_stats(flair_img, mask_img)

    return {'t1': t1_stats,
            't1ce': t1ce_stats,
            't2': t2_stats,
            'flair': flair_stats}

# Calculate the Tumor Volume 

In [5]:
dataset = 'Brats2023'
data_path = '../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
patient_ids = [f for f in listdir(data_path) if not isfile(join(data_path, f))]

image_intensities = {}
count = 0
for _id in patient_ids:
    print(_id)
    count += 1
    patient_id = _id.split('GLI-')[1]
#     image_intensity = calculate_intensity_mask_only(dataset, patient_id)
    image_intensity = calculate_intensity_tumor(dataset, patient_id)
#     image_intensity = calculate_intensity_tumor_whole_image(dataset, patient_id)
    
    image_intensities[_id] = image_intensity
#     if count == 2:
#         break
# image_intensities_df = pd.DataFrame.from_dict(image_intensities, 
#                                          orient = 'index', 
#                                          columns = ['t1', 
#                                                     't1ce', 
#                                                     't2', 
#                                                     'flair'])

# image_intensities_df.to_csv('../Results/Analysis_Results/intensity/GLI-Image_intensity_whole.csv')
# with open('../Results/Analysis_Results/intensity/GLI-Image_intensity_whole.pkl', 'wb') as handle:
#     pkl.dump(image_intensities, handle, protocol=pkl.HIGHEST_PROTOCOL)

BraTS-GLI-00324-000
BraTS-GLI-00442-000
BraTS-GLI-00456-000
BraTS-GLI-00318-000
BraTS-GLI-01012-000
BraTS-GLI-00481-000
BraTS-GLI-00495-000
BraTS-GLI-00640-000
BraTS-GLI-00126-000
BraTS-GLI-01238-000
BraTS-GLI-00132-000
BraTS-GLI-00654-000
BraTS-GLI-00045-001
BraTS-GLI-01204-000
BraTS-GLI-01210-000
BraTS-GLI-00668-000
BraTS-GLI-00683-000
BraTS-GLI-00697-000
BraTS-GLI-00734-000
BraTS-GLI-00052-000
BraTS-GLI-01358-000
BraTS-GLI-00046-000
BraTS-GLI-00708-000
BraTS-GLI-01416-000
BraTS-GLI-01370-000
BraTS-GLI-01364-000
BraTS-GLI-01402-000
BraTS-GLI-00085-000
BraTS-GLI-00694-001
BraTS-GLI-00680-001
BraTS-GLI-00469-001
BraTS-GLI-00250-000
BraTS-GLI-01172-000
BraTS-GLI-01166-000
BraTS-GLI-00293-000
BraTS-GLI-01199-000
BraTS-GLI-00483-001
BraTS-GLI-00286-000
BraTS-GLI-01198-000
BraTS-GLI-00292-000
BraTS-GLI-01167-000
BraTS-GLI-01173-000
BraTS-GLI-00523-000
BraTS-GLI-00251-000
BraTS-GLI-00537-000
BraTS-GLI-00084-000
BraTS-GLI-00090-000
BraTS-GLI-01365-000
BraTS-GLI-01403-000
BraTS-GLI-01417-000


BraTS-GLI-01516-000
BraTS-GLI-00608-000
BraTS-GLI-00031-001
BraTS-GLI-01502-000
BraTS-GLI-01264-000
BraTS-GLI-00191-000
BraTS-GLI-00185-000
BraTS-GLI-00807-000
BraTS-GLI-01338-000
BraTS-GLI-00026-000
BraTS-GLI-00740-000
BraTS-GLI-00032-000
BraTS-GLI-01304-000
BraTS-GLI-01462-000
BraTS-GLI-00768-000
BraTS-GLI-01476-000
BraTS-GLI-01310-000
BraTS-GLI-01489-000
BraTS-GLI-00797-000
BraTS-GLI-00542-000
BraTS-GLI-00230-000
BraTS-GLI-00556-000
BraTS-GLI-01660-000
BraTS-GLI-00218-000
BraTS-GLI-01106-000
BraTS-GLI-01112-000
BraTS-GLI-00581-000
BraTS-GLI-00594-000
BraTS-GLI-00580-000
BraTS-GLI-01113-000
BraTS-GLI-01661-000
BraTS-GLI-01107-000
BraTS-GLI-00219-000
BraTS-GLI-00231-000
BraTS-GLI-00557-000
BraTS-GLI-00543-000
BraTS-GLI-00796-000
BraTS-GLI-01488-000
BraTS-GLI-00782-000
BraTS-GLI-01477-000
BraTS-GLI-00636-001
BraTS-GLI-01311-000
BraTS-GLI-01305-000
BraTS-GLI-01463-000
BraTS-GLI-00033-000
BraTS-GLI-01339-000
BraTS-GLI-00999-000
BraTS-GLI-00806-000
BraTS-GLI-00184-000
BraTS-GLI-01503-000


BraTS-GLI-00837-000
BraTS-GLI-01297-000
BraTS-GLI-01283-000
BraTS-GLI-00823-000
BraTS-GLI-00758-000
BraTS-GLI-01446-000
BraTS-GLI-01320-000
BraTS-GLI-00607-001
BraTS-GLI-00613-001
BraTS-GLI-01334-000
BraTS-GLI-01452-000
BraTS-GLI-00764-000
BraTS-GLI-00002-000
BraTS-GLI-01308-000
BraTS-GLI-00016-000
BraTS-GLI-01485-000
BraTS-GLI-01491-000
BraTS-GLI-01122-000
BraTS-GLI-00228-000
BraTS-GLI-01136-000
BraTS-GLI-00572-000
BraTS-GLI-00214-000
BraTS-GLI-00599-000
BraTS-GLI-00598-000
BraTS-GLI-00201-000
BraTS-GLI-00567-000
BraTS-GLI-01137-000
BraTS-GLI-01123-000
BraTS-GLI-01490-000
BraTS-GLI-01484-000
BraTS-GLI-00017-000
BraTS-GLI-01309-000
BraTS-GLI-00765-000
BraTS-GLI-00003-000
BraTS-GLI-01335-000
BraTS-GLI-00612-001
BraTS-GLI-01453-000
BraTS-GLI-01447-000
BraTS-GLI-00759-000
BraTS-GLI-01321-000
BraTS-GLI-01282-000
BraTS-GLI-00836-000
BraTS-GLI-00188-000
BraTS-GLI-01296-000
BraTS-GLI-00605-000
BraTS-GLI-00611-000
BraTS-GLI-01269-000
BraTS-GLI-00177-000
BraTS-GLI-01241-000
BraTS-GLI-00639-000


BraTS-GLI-00063-000
BraTS-GLI-00077-000
BraTS-GLI-01369-000
BraTS-GLI-01427-000
BraTS-GLI-00739-000
BraTS-GLI-01341-000
BraTS-GLI-01355-000
BraTS-GLI-01433-000
BraTS-GLI-00048-001
BraTS-GLI-01209-000
BraTS-GLI-00117-000
BraTS-GLI-00103-000
BraTS-GLI-01235-000
BraTS-GLI-01221-000
BraTS-GLI-00659-000
BraTS-GLI-00498-000
BraTS-GLI-00301-000
BraTS-GLI-00510-001
BraTS-GLI-00329-000
BraTS-GLI-01037-000
BraTS-GLI-01023-000


In [61]:
with open('../Results/Analysis_Results/intensity/GLI-Image_intensity_tumor_vs_non_tumor.pkl', 'wb') as handle:
    pkl.dump(image_intensities, handle, protocol=pkl.HIGHEST_PROTOCOL)